# Lab 5 (Bonus) — Reranking in ES|QL: Precision After Recall

**Thesis:** Retrieval (FORK+FUSE) gets the right documents into the top-N — that's **recall**. A reranker fixes the *order at the very top* — that's **precision**. In ES|QL it's one pipe stage: `RERANK`.

## What you'll learn
- The two-stage pattern — recall (`FUSE`) then precision (`RERANK`) — in a single ES|QL query
- Two reranker architectures: **pointwise** cross-encoder (Jina v2) vs **listwise** (Jina v3)
- How to swap rerankers by changing one `inference_id`
- When a reranker earns its latency — and when to skip it

## The `RERANK` command
```esql
... | FUSE | SORT _score DESC | LIMIT 20
| RERANK ?q ON body WITH {"inference_id": ".jina-reranker-v3"}
| LIMIT 5
```
`RERANK` sends the query text + each candidate's `body` to the rerank endpoint and **overwrites `_score`** with the reranker's relevance score. Always `LIMIT` to a sane candidate count before it (it calls inference per row).

In [ ]:
# --- Workshop helpers (inline — same block across all ES|QL notebooks) ---
# ES|QL edition: every search runs through es.esql.query() instead of es.search().
# Defined inline so this notebook is self-contained and runs from the repo too.

import os, json, time
import requests
from elasticsearch import Elasticsearch

INDEX = "aiewf-workshop-docs"

ES_ENDPOINT = os.environ.get("ES_ENDPOINT")
ES_API_KEY  = os.environ.get("ES_API_KEY")
if not ES_ENDPOINT or not ES_API_KEY:
    raise ValueError(
        "Set ES_ENDPOINT and ES_API_KEY.\n"
        "  In Instruqt: pre-configured in the sandbox.\n"
        "  Re-running the repo: export ES_ENDPOINT=https://...  export ES_API_KEY=..."
    )

# request_timeout=120: RERANK and COMPLETION (Labs 4-5) call inference per row and
# can take several seconds — the default 10s would time out the LLM step.
es = Elasticsearch(ES_ENDPOINT, api_key=ES_API_KEY, request_timeout=120)

def esql(query, **params):
    """Run an ES|QL query with named parameters (?name in the query string).

    Usage:  esql(QUERY, q="securing cluster traffic")
    ES|QL named params take the form params=[{"name": value}, ...]. If your pinned
    client rejects named params, switch to positional `?` and params=[value, ...] —
    never f-string the query text in (injection + teaches the wrong pattern).
    """
    param_list = [{k: v} for k, v in params.items()] if params else None
    return es.esql.query(query=query, params=param_list, format="json")

def rows(resp):
    """Turn an ES|QL response ({columns, values}) into a list of dicts keyed by column."""
    cols = [c["name"] for c in resp["columns"]]
    return [dict(zip(cols, vals)) for vals in resp["values"]]

def show_esql(resp, fields=("id", "title", "summary"), score=True):
    """Pretty-print ES|QL rows as a ranked table (mirrors the DSL notebooks' show_hits)."""
    data = rows(resp)
    if not data:
        print("  (no rows)"); return
    for rank, r in enumerate(data, 1):
        cols = "  ".join(str(r.get(f, "")) for f in fields)
        sc = r.get("_score")
        s = f"  {sc:.4f}" if score and sc is not None else ""
        print(f"  #{rank:<2}{s}  {cols}")

print("✓ ES|QL helpers loaded")


POINTWISE_ID = ".jina-reranker-v2-base-multilingual"   # pointwise cross-encoder
LISTWISE_ID  = ".jina-reranker-v3"                       # listwise reranker

# RRF recall stage (FORK+FUSE), reused as the candidate generator.
Q_RRF = ("FROM aiewf-workshop-docs METADATA _score, _id, _index\n"
         "| FORK ( WHERE MATCH(body, ?q)          | SORT _score DESC | LIMIT 50 )\n"
         "       ( WHERE MATCH(body_semantic, ?q) | SORT _score DESC | LIMIT 50 )\n"
         "| FUSE | SORT _score DESC | LIMIT 8 | KEEP id, title, summary, _score")

def q_rerank(inference_id):
    """Two-stage pipeline: RRF recall -> RERANK precision with the given endpoint."""
    return (
        "FROM aiewf-workshop-docs METADATA _score, _id, _index\n"
        "| FORK ( WHERE MATCH(body, ?q)          | SORT _score DESC | LIMIT 50 )\n"
        "       ( WHERE MATCH(body_semantic, ?q) | SORT _score DESC | LIMIT 50 )\n"
        "| FUSE | SORT _score DESC | LIMIT 8\n"
        '| RERANK ?q ON body WITH {"inference_id": "%s"}\n'
        # RERANK overwrites _score but does NOT reorder rows -- without this SORT,
        # rows keep their pre-rerank (RRF) order and the reranking is invisible.
        "| SORT _score DESC | LIMIT 8 | KEEP id, title, summary, _score" % inference_id
    )

print("✓ Lab 5 helpers loaded")

## What reranking *is* — a second stage on top of retrieval

Retrieval is optimized for **recall at speed**: scan the whole corpus, return a good top-N fast. It scores query and document somewhat independently (BM25 term stats; a query vector vs precomputed doc vectors). That's why the *right* doc is usually in the top 10 — but not always at #1.

A **reranker** is a different kind of model. It reads the **query and a candidate document together** and outputs a relevance score for that exact pair. Far more accurate per comparison — and far more expensive, so you only run it on the top-N that retrieval already narrowed down.

```
query ──► [ FORK | FUSE ]  ──►  top 8 candidates  ──►  [ RERANK ]  ──►  reordered top-K
            recall (fast)                                precision (accurate)
```

## The two TYPES of reranker — pointwise vs listwise

- **Pointwise cross-encoder** (Jina Reranker **v2**): scores each `(query, doc)` pair **independently**. Document order doesn't affect any single score. Simple and parallelizable.
- **Listwise** (Jina Reranker **v3**): scores the **whole candidate set jointly** in one pass, so the model can compare candidates against each other — useful when several near-duplicates compete for the top slots.

You swap between them in ES|QL by changing one `inference_id`. We'll run both head-to-head.

In [ ]:
# Confirm both rerank endpoints exist on this project.
eps = es.inference.get().body.get("endpoints", [])
rerankers = [e["inference_id"] for e in eps if e.get("task_type") == "rerank"]
print("rerank-task endpoints available:")
for e in rerankers:
    print(f"  {e}")
for need in (POINTWISE_ID, LISTWISE_ID):
    print(f"  {'✓' if need in rerankers else '⚠ MISSING'}  {need}")

## Before / after — reranking breaks a near-tie

Start with a query where RRF's recall stage produces a genuine **near-tie** at the top — two plausible answers barely separated by score — then add `RERANK` and watch a cross-encoder, reading query+doc jointly, make a confident call. `reduce storage cost for old logs` surfaces both the data-tiers doc (`doc-041`) and the ILM overview (`doc-017`) almost neck-and-neck — and the two reranker *types* don't even agree on which one should win.

In [ ]:
q = "reduce storage cost for old logs"   # doc-041 (data tiers) & doc-017 (ILM) near-tie
print(f"QUERY: {q!r}\n")
print("STAGE 1 -- RRF recall (doc-041 and doc-017 are a near-tie for #1):")
show_esql(esql(Q_RRF, q=q))
print(f"\nSTAGE 1+2 -- after RERANK with {LISTWISE_ID}:")
try:
    show_esql(esql(q_rerank(LISTWISE_ID), q=q))
    print("\n_score is now the reranker's relevance score, and the row order changed")
    print("because we re-sorted on it (RERANK alone does not reorder). Listwise reads")
    print("doc-041 and doc-017 against each other and flips its pick to the ILM overview --")
    print("pointwise (head-to-head below) actually agrees with RRF and keeps doc-041 on top.")
    print("Neither reranker is 'wrong'; they weigh the near-tie differently.")
except Exception as e:
    print(f"⚠ RERANK unavailable: {str(e)[:150]}")


## …and sharpen an already-decisive stage

Sometimes RRF already gets the right doc to #1, but only by a hair over a close second. The reranker — reading query+doc together — doesn't need to *reorder* anything here; it turns a fragile lead into a decisive one. `cluster.routing.allocation.enable` → `doc-008`.

In [ ]:
q = "cluster.routing.allocation.enable"   # target doc-008 -- RRF already has it #1, but barely
print(f"QUERY: {q!r}\n")
print("STAGE 1 -- RRF recall (doc-008 is #1, but only a hair ahead of doc-023):")
show_esql(esql(Q_RRF, q=q))
print(f"\nSTAGE 1+2 -- after RERANK with {LISTWISE_ID}:")
try:
    show_esql(esql(q_rerank(LISTWISE_ID), q=q))
except Exception as e:
    print(f"⚠ RERANK unavailable: {str(e)[:150]}")
print("\nThe rank does not change -- doc-008 was already right -- but the score gap to")
print("the runner-up widens dramatically. That is the reranker earning its keep even when")
print("recall already nailed it: a fragile #1 becomes a confident one.")


## Pointwise vs listwise — head to head

Same query, same candidates, swap only the `inference_id`. The corpus has a paraphrase pair: `doc-001` (SAML *authentication*) and `doc-002` (*authorization* / role-mapping). Both rerankers will rank `doc-001` #1 for `user cannot authenticate` — the interesting signal is **where `doc-002` lands**. Listwise, scoring the whole set jointly, tends to push the authorization doc down (the query is about *login* failure, not post-login roles).

In [ ]:
q = "user cannot authenticate"
print(f"QUERY: {q!r}")
print("Watch where doc-002 (authorization) lands in each — doc-001 (SAML auth) should be #1 in both.\n")

for label, eid in [("POINTWISE v2", POINTWISE_ID), ("LISTWISE  v3", LISTWISE_ID)]:
    print(f"{label}  ({eid}):")
    try:
        show_esql(esql(q_rerank(eid), q=q))
    except Exception as e:
        print(f"  ⚠ unavailable — {str(e)[:120]}")
    print()

print("Pointwise scores each doc alone, so doc-002 can rank high on its own merits.")
print("Listwise scores the set together and tends to drop doc-002 — it's about authorization,")
print("while the query is about authentication. Same candidates, different orderings.")

## Which reranker — and should you rerank at all?

**Pointwise (v2)** — cheaper, parallelizable, order-independent. Great default for re-scoring a top-N where candidates are mostly distinct.

**Listwise (v3)** — scores the set jointly; wins when several near-duplicates compete for the top slots and relative judgments matter. More expensive.

**When to add a reranker at all:**
- ✅ The top-1/top-3 *order* matters (RAG context window, a "best answer" UI, an agent that reads only the first result).
- ✅ Stage-1 recall is good but precision-at-1 is shaky.
- ❌ Skip it when FUSE already nails #1, or when latency budget is tight and "good enough in the top 5" is fine — every reranked query is an extra inference call per candidate.

**The ES|QL payoff:** recall *and* precision live in one statement. Change `RERANK ?q ON body WITH {...}` and the whole stage swaps — no separate rerank service, no glue code.

---
*That's the bonus lab. You've now expressed every stage of a modern retrieval pipeline — semantic, BM25, RRF/linear fusion, reranking, and LLM synthesis — entirely in ES|QL.*